# Scraper Tempat Makan & Wisata Viral di Indonesia (6 Bulan Terakhir)

Notebook ini melakukan scraping berita dari **Google News RSS** (bahasa Indonesia) untuk mencari artikel tentang tempat makan/wisata yang sedang viral dalam 6 bulan terakhir, lalu mengekstrak:

- Nama tempat
- Lokasi (kelurahan, kecamatan, kota/kabupaten, provinsi) + latitude/longitude (via geocoding OpenStreetMap Nominatim)
- Sejak kapan viral (perkiraan dari tanggal berita/isi artikel)
- Kenapa viral (kalimat yang mengandung alasan viral)
- Sumber berita (judul, link, tanggal publish)

## Catatan penting
- Ekstraksi nama tempat, lokasi, dan alasan viral memakai **heuristik berbasis regex**, bukan NLP/NER penuh, karena berita berbahasa Indonesia sangat bervariasi formatnya. Hasil disarankan **dicek/dibersihkan manual** sebelum dipakai untuk keperluan resmi.
- Geocoding memakai **Nominatim (OpenStreetMap)** yang gratis tapi ada batas 1 request/detik dan mewajibkan `user_agent` yang jelas. Untuk kebutuhan volume besar/produksi, pertimbangkan pakai Google Maps Geocoding API (berbayar) untuk akurasi lebih baik.
- Gunakan data ini untuk riset/analisis internal. Selalu cek Terms of Service situs sumber sebelum melakukan scraping berskala besar.

## 1. Install & Import Library

In [ ]:
%pip install feedparser requests beautifulsoup4 lxml geopy pandas python-dateutil tqdm openpyxl pytrends googlenewsdecoder -q

In [1]:
import json
import re
import time
import urllib.parse
from datetime import datetime, timedelta, timezone

import feedparser
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from googlenewsdecoder import gnewsdecoder
from tqdm.auto import tqdm

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

c:\Users\Agus Zainal Arifin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Konfigurasi

Ubah `KEYWORDS` atau `MONTHS_BACK` sesuai kebutuhan.

In [2]:
BASE_KEYWORDS = [
    "tempat makan viral",
    "kuliner viral",
    "cafe viral",
    "restoran viral",
    "warung viral",
    "jajanan viral",
    "street food viral",
    "wisata viral",
    "destinasi wisata viral",
    "spot wisata viral",
    "tempat wisata baru viral",
    "museum viral",
    "galeri seni viral",
    "pameran viral",
    "event viral",
    "festival viral",
    "pasar viral",
    "pasar malam viral",
    "toko viral",
    "toko oleh-oleh viral",
    "toko barang unik viral",
    "thrift shop viral",
    "pasar loak viral",
    "pusat oleh-oleh viral",
    "tempat belanja viral",
    "hidden gem Indonesia",
    "tempat hits Indonesia",
]

PROVINCES = [
    "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Kepulauan Riau",
    "Jambi", "Sumatera Selatan", "Kepulauan Bangka Belitung", "Bengkulu", "Lampung",
    "DKI Jakarta", "Jawa Barat", "Banten", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur",
    "Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat",
    "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara",
    "Sulawesi Utara", "Gorontalo", "Sulawesi Tengah", "Sulawesi Barat", "Sulawesi Selatan",
    "Sulawesi Tenggara", "Maluku", "Maluku Utara", "Papua", "Papua Barat",
    "Papua Selatan", "Papua Tengah", "Papua Pegunungan", "Papua Barat Daya",
]

BIG_CITIES = [
    "Jakarta Selatan", "Jakarta Pusat", "Jakarta Barat", "Jakarta Timur", "Jakarta Utara",
    "Surabaya", "Bandung", "Medan", "Yogyakarta", "Sleman", "Semarang", "Makassar",
    "Palembang", "Depok", "Bekasi", "Tangerang", "Bogor", "Malang", "Denpasar",
    "Bandar Lampung", "Padang", "Pekanbaru", "Banjarmasin", "Balikpapan", "Samarinda",
    "Pontianak", "Manado", "Jayapura", "Solo", "Cirebon", "Tasikmalaya", "Batam",
]

REGIONAL_CATEGORIES = [
    "tempat makan viral",
    "wisata viral",
    "museum viral",
    "toko viral",
    "tempat belanja viral",
    "pasar viral",
    "event viral",
]

LOCATION_KEYWORDS = [
    f"{category} {region}"
    for region in PROVINCES + BIG_CITIES
    for category in REGIONAL_CATEGORIES
]
KEYWORDS = list(dict.fromkeys(BASE_KEYWORDS + LOCATION_KEYWORDS))

MONTHS_BACK = 6
MAX_ITEMS_PER_KEYWORD = 20
REQUEST_DELAY = 1.0

CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=30 * MONTHS_BACK)
print("Total keyword:", len(KEYWORDS))
print("Ambil artikel sejak:", CUTOFF_DATE.date())

Total keyword: 517
Ambil artikel sejak: 2026-04-01


In [3]:
KEYWORDS

['tempat makan viral',
 'kuliner viral',
 'cafe viral',
 'restoran viral',
 'warung viral',
 'jajanan viral',
 'street food viral',
 'wisata viral',
 'destinasi wisata viral',
 'spot wisata viral',
 'tempat wisata baru viral',
 'museum viral',
 'galeri seni viral',
 'pameran viral',
 'event viral',
 'festival viral',
 'pasar viral',
 'pasar malam viral',
 'toko viral',
 'toko oleh-oleh viral',
 'toko barang unik viral',
 'thrift shop viral',
 'pasar loak viral',
 'pusat oleh-oleh viral',
 'tempat belanja viral',
 'hidden gem Indonesia',
 'tempat hits Indonesia',
 'tempat makan viral Aceh',
 'wisata viral Aceh',
 'museum viral Aceh',
 'toko viral Aceh',
 'tempat belanja viral Aceh',
 'pasar viral Aceh',
 'event viral Aceh',
 'tempat makan viral Sumatera Utara',
 'wisata viral Sumatera Utara',
 'museum viral Sumatera Utara',
 'toko viral Sumatera Utara',
 'tempat belanja viral Sumatera Utara',
 'pasar viral Sumatera Utara',
 'event viral Sumatera Utara',
 'tempat makan viral Sumatera Bar

## 3. Ambil Daftar Berita dari Google News RSS

In [5]:
def fetch_google_news(keyword: str, max_items: int = 25):
    """Ambil daftar berita dari Google News RSS untuk sebuah keyword (bahasa Indonesia)."""
    query = urllib.parse.quote(keyword)
    url = f"https://news.google.com/rss/search?q={query}&hl=id&gl=ID&ceid=ID:id"
    feed = feedparser.parse(url)

    items = []
    for entry in feed.entries[:max_items]:
        try:
            pub_date = dateparser.parse(entry.published)
            if pub_date.tzinfo is None:
                pub_date = pub_date.replace(tzinfo=timezone.utc)
        except Exception:
            pub_date = None

        if pub_date is not None and pub_date < CUTOFF_DATE:
            continue  # lewati berita yang lebih lama dari batas waktu

        source = entry.get("source", {}).get("title") if hasattr(entry, "get") else None
        items.append(
            {
                "keyword": keyword,
                "title": entry.title,
                "link": entry.link,
                "published": pub_date,
                "source": source or getattr(getattr(entry, "source", None), "title", None),
            }
        )
    return items


all_news = []
for kw in tqdm(KEYWORDS, desc="Fetch RSS per keyword"):
    all_news.extend(fetch_google_news(kw, MAX_ITEMS_PER_KEYWORD))
    time.sleep(REQUEST_DELAY)

df_news = pd.DataFrame(all_news).drop_duplicates(subset="link").reset_index(drop=True)
print(f"Total artikel ditemukan: {len(df_news)}")
df_news.head()

Fetch RSS per keyword: 100%|██████████| 517/517 [16:14<00:00,  1.88s/it]

Total artikel ditemukan: 2473


,keyword,title,link,published,source
0,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
1,tempat makan viral,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legend...,https://news.google.com/rss/articles/CBMitgFBV...,2026-09-26 08:00:30+00:00,detikFood
2,tempat makan viral,Rekomendasi 7 Tempat Makan Ramah Kantong di Jo...,https://news.google.com/rss/articles/CBMiwgFBV...,2026-09-23 06:38:00+00:00,RRI.co.id
3,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
4,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co


In [7]:
df_news = pd.read_csv("df_news_extended_1_249.csv")
# remove last 50 row 
df_news = df_news[:-50]
# df_news.keyword.value_counts()
df_news

,keyword,title,link,published,source
0,tempat makan viral,Media Sosial Mengubah Cara Kita Memilih Tempat...,https://news.google.com/rss/articles/CBMifkFVX...,2026-09-25 08:00:00+00:00,Jatimnet.com
1,tempat makan viral,Awas Kalap! Ini 7 Tempat Makan Soto Mie Legend...,https://news.google.com/rss/articles/CBMitgFBV...,2026-09-26 08:00:30+00:00,detikFood
2,tempat makan viral,Rekomendasi 7 Tempat Makan Ramah Kantong di Jo...,https://news.google.com/rss/articles/CBMiwgFBV...,2026-09-23 06:38:00+00:00,RRI.co.id
3,tempat makan viral,10 Tempat Makan Viral di Bandung 2026 - nibble.id,https://news.google.com/rss/articles/CBMiaEFVX...,2026-06-09 07:00:00+00:00,nibble.id
4,tempat makan viral,"TikTok Food Fest 2026 Hari Terakhir di GBK, Ad...",https://news.google.com/rss/articles/CBMidEFVX...,2026-09-27 10:22:00+00:00,Aktualita.co
...,...,...,...,...,...
2418,tempat makan viral Batam,5 Tempat Makan Siang Enak di Batam yang Cocok ...,https://news.google.com/rss/articles/CBMixwFBV...,2026-06-11 07:00:00+00:00,Jawa Pos
2419,tempat makan viral Batam,Jelajah Kuliner Batam: 5 Tempat Makan Baru den...,https://news.google.com/rss/articles/CBMisAFBV...,2026-07-06 07:00:00+00:00,Batam Pos
2420,tempat makan viral Batam,"Pilihan Tempat Makan Siang Hits di Batam, Saji...",https://news.google.com/rss/articles/CBMiswFBV...,2026-05-17 07:00:00+00:00,batam.inews.id
2421,tempat makan viral Batam,"Wisata Kuliner Batam Makin Ramai, 5 Tempat Kul...",https://news.google.com/rss/articles/CBMiygFBV...,2026-05-29 07:00:00+00:00,Jawa Pos


## 4. Ambil Isi Artikel (Full Text)

Link dari Google News RSS adalah redirect, kita ikuti sampai ke halaman aslinya lalu ambil teks paragraf.

In [ ]:
def resolve_and_fetch(url: str, timeout: tuple[int, int] = (5, 15)):
    """Decode URL Google News lalu ambil HTML halaman artikel sumber."""
    source_url = url
    if "news.google.com" in url:
        try:
            decoded = gnewsdecoder(url, interval=1, timeout=10)
            if decoded.get("success") and decoded.get("decoded_url"):
                source_url = decoded["decoded_url"]
        except Exception:
            pass

    try:
        resp = requests.get(source_url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        return resp.url, resp.text
    except requests.RequestException:
        return None, None


def extract_article_text(html: str) -> str:
    """Ambil isi artikel dari JSON-LD, meta description, paragraf, atau body HTML."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")

    # Banyak situs berita menyimpan isi artikel di JSON-LD, bukan tag p.
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            payload = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        candidates = payload if isinstance(payload, list) else [payload]
        for item in candidates:
            if isinstance(item, dict) and isinstance(item.get("articleBody"), str):
                article_body = re.sub(r"\s+", " ", item["articleBody"]).strip()
                if len(article_body) >= 100:
                    return article_body

    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    meta_text = meta.get("content", "").strip() if meta else ""

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [p for p in paragraphs if len(p) >= 20]
    paragraph_text = " ".join(paragraphs)
    if len(paragraph_text) >= 100:
        return paragraph_text
    if len(meta_text) >= 40:
        return meta_text

    body_text = soup.get_text(" ", strip=True) if soup.body else ""
    return re.sub(r"\s+", " ", body_text).strip()


article_texts = []
final_urls = []
i = 1
for link in tqdm(df_news["link"], desc="Fetch isi artikel"):
    if i > 2410: break
    i += 1
    print("Processing article number:", i)
    print("Fetching link:", link)
    final_url, html = resolve_and_fetch(link)
    final_urls.append(final_url)
    article_texts.append(extract_article_text(html))
    print("panjang artikel:", len(article_texts[-1]))
    time.sleep(REQUEST_DELAY)

df_news["final_url"] = final_urls
df_news["article_text"] = pd.Series(article_texts, index=df_news.index, dtype="string")
df_news["article_text_length"] = df_news["article_text"].str.len().fillna(0).astype(int)
print(f"Artikel dengan isi berhasil diambil: {(df_news['article_text_length'] >= 100).sum()} dari {len(df_news)}")
print(f"Fetch gagal/kosong: {df_news['article_text'].eq('').sum()}")
df_news = df_news[df_news["article_text_length"] >= 100].reset_index(drop=True)
df_news.to_csv("df_news_extended.csv", index=False)

Fetch isi artikel:   0%|          | 0/2423 [00:00<?, ?it/s]

Fetching link: https://news.google.com/rss/articles/CBMifkFVX3lxTE9qZDRrbTc5elBjLU5ra0pZVnp1M21wMFRZd1NydUpGY2xlbm5vMktZZEJXNnhfcVZVWG1uRHJUSF9Dc0xydVdoLWItQ0RMMEd2dXQyMUNWeFB0QjNNZU02bmg2OTlOR0JtZVRBWW5pUWhUc0ZZcTdDbjZobmtNUQ?oc=5
panjang artikel: 7110


Fetch isi artikel:   0%|          | 1/2423 [00:04<3:09:17,  4.69s/it]

Fetching link: https://news.google.com/rss/articles/CBMitgFBVV95cUxOYk92ckFweFFab1BZMHdPUTB2U2NXeE5vOVpHOUJfbXVqUm5YeHNHVjVUdUYxT2x2VVZtVnBHeEdTYmVBZXI3Y2xpWGRLMWcwMnJJNlduRUtqREJLbzhOQWpPMHhaRXc4cjBXRUwwbWZVUlViWFduN2plTU93UF9MVDRuRmZ0SW5rR2pYbmNUQjc1MG00bnJpSGtMN3ZIcVdhRTd1MXctcy1WOXhraWhiR2xfRVQ0Z9IBuwFBVV95cUxQQjFXOGp4UXEyUlp5aDl4dFFYN3VnQnBFN0E3akFRQ2ljdE1rcndCMkVldVhWcHFDekQwU3J2U0NNZzB1dGFGQWlxWGRWUXVPNU5OaUt0NmRIX3RaT1JjSFoxZU9RQXpXaTdYcXN4SEpZWTRSbnFUNHIxYk1ZUVM2SExaemdKNC1MX0tqemlKY1phWi1XajR2VVlMU0xtNGRJNzVRdG5VbUVWUjlqVktDNlYyd0p2eG1MZzFj?oc=5
panjang artikel: 2855


Fetch isi artikel:   0%|          | 2/2423 [00:09<3:11:15,  4.74s/it]

...

Fetching link: https://news.google.com/rss/articles/CBMilwFBVV95cUxOTGdUWEhNSEZYNE5xT1dPZ0I1aXo5MUxBZVQ5Z2h0MjRGWF9SZGFkVWhiel85TktpSEN2X3RaNEJSbUVaRW1xcEphZUw1NW43d1V4czBOM1RFMTRkQzNfVERWVm1KUHNCM0dXSFJ2Z3FyNFFRVmJjclhzNHRYZzV5cHhZNU51VnA4ZFdYcHNud3ptOEE1R3l30gGcAUFVX3lxTE5EeW9UMUZlXzdBVVNXMDk1cHBLNFBNbFJ6R213dW15ZUZnYi1ldFNUMElKSzlfV1lwSEtYS1dGVFpncVZsQm10QkxvcnVZcE10SGt1RkY3MzRaZFdVWjBNY0ZpUjNNd09VdzFuQXRCUGRmckh4cjdXVkE3Q0NYTGlLQkxITE1DS0RKSHFnSjRaTkVYMEVvYU8tQUZXdQ?oc=5
panjang artikel: 403


Fetch isi artikel:   8%|▊         | 189/2423 [16:24<3:05:44,  4.99s/it]

Fetching link: https://news.google.com/rss/articles/CBMi6wFBVV95cUxQd21ETlhETnFzSnltLWt1c2RWa0dfMW9qUXdSSVB5alprRXJJN2hNc0pFX3N4dGtjbVBlYlNQSWtmZDc1Tk9sX09IcWc3c2ZzUDVKa3V1Z3JobUI5MjUyMnZRMGlMOTA1WUhmbGs1UC1Vd0I0VFN6TUFlUG5aYmdzM3lZUGgweFdVbzdfQ285X05kODNEblFzbnFPVGcwUktnWWZyUGxMUXB3dW1qSXQxMFRfakZjeVRzNmZOdEo3N2x2ZGRRekNZN0Z2dmJBQjBwUW1SRHBxZzNWai1wTEdIZHJQczd5QnlzNWhN0gHwAUFVX3lxTE5mT3pjYU1lMkpIMG1LcGhvOGE5LTJfUFdFTDV4V2hjZDJEeW9FOXJxM1RZTXkxbHdBVUZGVk9kSEJ6czJkU2JPSDNvQVE3d0hGaGcxQm1pYmpiRDlIVG5xNlRvekhhNDkxLVNHTEwwOGNoX3lRRzRIMEttMVppSnk2SHRfMWRrVmUtTk1yZ0hfRUR1VU0tc3BCVHIyMEVXRDdzLV9PcG80V01RTHkwVjgwcW5TdmdtMFZrZ3E3S0RESUF3dENXV1lwYXRPVjlUMmstWUY0a0lsNGRuLU1kbm04dXF0VGs2OGR0TXAzRmZMNA?oc=5
panjang artikel: 117


Fetch isi artikel:   8%|▊         | 190/2423 [16:30<3:09:49,  5.10s/it]

Fetching link: https://news.google.com/rss/articles/CBMiiAFBVV95cUxQUEN1eFdNcW9mRm1kVnp4UGNiSEEzSlppUEZua0FRcmZ2bFZMM01tckpUTnpaUlN1QmhvQjhCSHIwYUNMNkxrOGJEeDNleDY0LW9xR3paQXd6R0E2ejBmY0FwLW5uSUZ1akJNZXgwUF9TTU1IY2lVU1p6YjBZZURZTk40cFdsT0ot?oc=5


In [ ]:
# df_news.to_csv("df_news_extended_1_249.csv", index=False)

In [9]:
def resolve_and_fetch(url: str, timeout: tuple[int, int] = (5, 15)):
    """Decode URL Google News lalu ambil HTML halaman artikel sumber."""
    source_url = url
    if "news.google.com" in url:
        try:
            decoded = gnewsdecoder(url, interval=1, timeout=10)
            if decoded.get("success") and decoded.get("decoded_url"):
                source_url = decoded["decoded_url"]
        except Exception:
            pass

    try:
        resp = requests.get(source_url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        return resp.url, resp.text
    except requests.RequestException:
        return None, None


def extract_article_text(html: str) -> str:
    """Ambil isi artikel dari JSON-LD, meta description, paragraf, atau body HTML."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")

    # Banyak situs berita menyimpan isi artikel di JSON-LD, bukan tag p.
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            payload = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        candidates = payload if isinstance(payload, list) else [payload]
        for item in candidates:
            if isinstance(item, dict) and isinstance(item.get("articleBody"), str):
                article_body = re.sub(r"\s+", " ", item["articleBody"]).strip()
                if len(article_body) >= 100:
                    return article_body

    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    meta_text = meta.get("content", "").strip() if meta else ""

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [p for p in paragraphs if len(p) >= 20]
    paragraph_text = " ".join(paragraphs)
    if len(paragraph_text) >= 100:
        return paragraph_text
    if len(meta_text) >= 40:
        return meta_text

    body_text = soup.get_text(" ", strip=True) if soup.body else ""
    return re.sub(r"\s+", " ", body_text).strip()


article_texts = []
final_urls = []
i = 1
for link in tqdm(df_news["link"], desc="Fetch isi artikel"):
    if i < 250:
        i += 1
        continue
    print("Fetching link:", link)
    final_url, html = resolve_and_fetch(link)
    final_urls.append(final_url)
    article_texts.append(extract_article_text(html))
    print("panjang artikel:", len(article_texts[-1]))
    time.sleep(REQUEST_DELAY)

df_news["final_url"] = final_urls
df_news["article_text"] = pd.Series(article_texts, index=df_news.index, dtype="string")
df_news["article_text_length"] = df_news["article_text"].str.len().fillna(0).astype(int)
print(f"Artikel dengan isi berhasil diambil: {(df_news['article_text_length'] >= 100).sum()} dari {len(df_news)}")
print(f"Fetch gagal/kosong: {df_news['article_text'].eq('').sum()}")
df_news = df_news[df_news["article_text_length"] >= 100].reset_index(drop=True)
df_news.to_csv("df_news_extended.csv", index=False)

Fetch isi artikel:   0%|          | 0/2473 [00:00<?, ?it/s]

Fetching link: https://news.google.com/rss/articles/CBMizwFBVV95cUxNSGFiVnlfaUZyU2ZzVFZvTkdRTURaNnNiSGxHS0JVUFJRYVFQNXVwVkJIU1NJS1lWNXRVNVJ0WTFVM0dnUXhEMDNHZXdCQ2FWeFItMzlWaDcwUDA3bHVDVlp0NGxPVDMxSTlvWW5RVWJZSWxmMGM5elpkSTBydFJ2NTYtcG9EV1JnZ08yamlDM3BzbE1GVTJIX042b0hVbV94TmhscXRybVNvVFhXai0tNVg2UXlqYWE2Z1hrVU5YSmd4bHJId0ZSc1hKZDY3MnPSAdQBQVVfeXFMTzltZ0l6QVpyeHJKNTQ1LU5EUzdMSWZLN0dacFRaT29ob0oySEdTRjVIR2RLV0ZHQWMwczZGVXNFOHRac2FmRWo5dUhsa3M5aTV5QUd4WkpDUFpEbXhxMEliZ0oyNi11UXdZemYzTUdOOTJiQ3dqaGlTb1hVRjNHX1UxSXhJeC1WRVJpRzBnNXR6VllkVkJRelZkT0pESEtiQVNjWlBsZGUyLVY1dU1INnhYd0hjYWhheFYwRU0zRERSaFBlbzFqb0pxZEJqejlpekh4eGQ?oc=5
panjang artikel: 4802


Fetch isi artikel:  10%|█         | 250/2473 [00:03<00:33, 65.98it/s]

Fetching link: https://news.google.com/rss/articles/CBMiqgFBVV95cUxNWldUREVla1dsaVB4bnpkUWpRMHZXRG81REV5OHA4VjktbU5nVjFDYWY2WXdGWUZXRTRFanN6eUJtYWg2YVBiYXpFVUVMQVpxSnhkcDZ5cXdfRUM5R0NsdnRVTTBVSEFNd2hzNFcwTHoxb1E0S055X1EtR3JsTVh2elluTTlVbHFXRVNuSzU3SjljbGRYSm55NlFaN3gwaGs3TTczbTd1dGtnZ9IBqAFBVV95cUxNbXRNYy1IS1E4ejZIZ0RXZnhMM1JJdldIWjV1ei1PY0hfQUctUmZ2ejlkcXFiT1RRcFk4bUgtNDh2Nmo4V05HRHczV0Jwd3hUUVV0a1AtSkpObWNGM29HcDRmNTFxT2ZwNkgyOXpSNXVla1FHNjVjQmtBMlctRW5SaFBUdW9TSnBLXzhiTVZpdEh1X3I5ajNzaWkxaEpZdWVTb0otSElDbVg?oc=5
panjang artikel: 2775
Fetching link: https://news.google.com/rss/articles/CBMiugFBVV95cUxQcWpQc3dabzZwUVVtMjhwX2FzeVJBeGJOd1NUQzNmXzlnZGozbTZieDl1VVFGUlAzdnJHUkNRaFE5MmRfc29hM3ZsSnBPQXZoSHFfeFVjTHVVdnRaWlBpeklOdDNpVFZla1hqUHctLXB1Ny11VWI2UjVvZDNsX1hvTDFBQ3k0Nk5EUGNqLVJVbDZEWXh6LW9JVmRpNUFZVElGdFZ5dElZUkE5SGwwS2kwc0kzQnZDMkZIMHfSAb8BQVVfeXFMT3loVmFjUTJ2a09wd2hEa3MteE5lOE5NSUlMYVp6WVE5bXVWbk1GYndHd0JQajZyQ3dFcExObUg3aXYyYjV1bzFfdFc5bTBNbHlqQ0RhOEJ5Mk5IZVFYdXJPd2FFYzIzMzlNNUFKc

Fetch isi artikel:  10%|█         | 250/2473 [00:18<00:33, 65.98it/s]

...

Fetching link: https://news.google.com/rss/articles/CBMisgFBVV95cUxQc1NTTlhZU05WVndLZTdFQTRKVFJZRjJDdWdZVnpwNGVWa2RFSEZtWlVOSFQwMlB4clJaT093VWp5UWd1Q1E5NEw1VEVDa2l3UzJoZlhDZUpFMDM5OWN0aWhFbmc1TWhEbFVPZlY2Tk85bjV2LTBQQW5CTlg2dXJCVVZsdTlVUTlhdEFqOEp3RGFRRXFlM1lEQXpwbVBxV0EzYVNfUzBxcnk5TktjNWJnMWpR0gG3AUFVX3lxTE96ajM1akd2TzBzWDZ0SURMaGFDRjBpMUlkLU9NdTAtRmppdEMwYzA3RERzM1dMNEhCN3ZyR0VYZ3N5SzFidHQ2aTRmQlZlSTZpQkZJWEZWX2dJUDRoYXJKZVVvS19OenN0dXFjV3BZNFNaUmcwckIxYkcyVldfdWZEa1VsQ1V6RTZNVXdvdXFYbXhlanc1TFJUM3hOY2FVTk9XcURfYXM1aTJhN0tuaHpFTkUyY1FvVQ?oc=5
panjang artikel: 1977


Fetch isi artikel:  30%|██▉       | 730/2473 [38:48<2:11:51,  4.54s/it]

Fetching link: https://news.google.com/rss/articles/CBMipgFBVV95cUxPZWMtT0pzQnlzNEdKVFRIbF9uN3Q1YnJZTGg4Mnh1bVdubXBGZDJBSUZ4YTZuc2RXejhheEVrNzJxM1RrcVBtQWYzdnFGNTRFaElmak1Bc3BPVkZCeUpNbHAtdGlDa3BOY3NqU1JmWEZYYmg1a3ZTVXBlbnFQU21fYnRlNVJvaC1wemZsNUg4STU0dEdGNUY3aUx0S05wWVMxdzk5Wldn0gGrAUFVX3lxTE42ZFJfMEFmbm9td1dGU01TVkpmQjdFc0p3TEI2QjVTUzltNHI0OHIxcThkNWt1MkN6ZEJBWjl2WU9US3E1eVJ2eGwtY3FMdEpmREx2ejdJYWg2MUhBSXhOSEhRaF9Rcm1hcHFfZ21wZEpnQS1YUUs2SDBXQXhfWkxnS2FHaHRiZkQ0SG5wb2x6RGdJbG1SdG1FVWZ5bkNyS3ZEY183YnVVZGFpcw?oc=5


Fetch isi artikel:  30%|██▉       | 730/2473 [38:50<1:32:45,  3.19s/it]


KeyboardInterrupt: 

In [10]:
df_news.to_csv("df_news_extended_250_503.csv", index=False)

In [ ]:
def resolve_and_fetch(url: str, timeout: tuple[int, int] = (5, 15)):
    """Decode URL Google News lalu ambil HTML halaman artikel sumber."""
    source_url = url
    if "news.google.com" in url:
        try:
            decoded = gnewsdecoder(url, interval=1, timeout=10)
            if decoded.get("success") and decoded.get("decoded_url"):
                source_url = decoded["decoded_url"]
        except Exception:
            pass

    try:
        resp = requests.get(source_url, headers=HEADERS, timeout=timeout, allow_redirects=True)
        resp.raise_for_status()
        return resp.url, resp.text
    except requests.RequestException:
        return None, None


def extract_article_text(html: str) -> str:
    """Ambil isi artikel dari JSON-LD, meta description, paragraf, atau body HTML."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "lxml")

    # Banyak situs berita menyimpan isi artikel di JSON-LD, bukan tag p.
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            payload = json.loads(script.string or script.get_text())
        except (TypeError, json.JSONDecodeError):
            continue
        candidates = payload if isinstance(payload, list) else [payload]
        for item in candidates:
            if isinstance(item, dict) and isinstance(item.get("articleBody"), str):
                article_body = re.sub(r"\s+", " ", item["articleBody"]).strip()
                if len(article_body) >= 100:
                    return article_body

    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    meta_text = meta.get("content", "").strip() if meta else ""

    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "noscript"]):
        tag.decompose()
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [p for p in paragraphs if len(p) >= 20]
    paragraph_text = " ".join(paragraphs)
    if len(paragraph_text) >= 100:
        return paragraph_text
    if len(meta_text) >= 40:
        return meta_text

    body_text = soup.get_text(" ", strip=True) if soup.body else ""
    return re.sub(r"\s+", " ", body_text).strip()


article_texts = []
final_urls = []
i = 1
mulai = 504
batas = 249

for link in tqdm(df_news["link"], desc="Fetch isi artikel"):
    if i < mulai:
        i += 1
        continue
    if i > mulai + batas:
        break
    print("Fetching link:", link)
    final_url, html = resolve_and_fetch(link)
    final_urls.append(final_url)
    article_texts.append(extract_article_text(html))
    print("panjang artikel:", len(article_texts[-1]))
    time.sleep(REQUEST_DELAY)

df_news["final_url"] = final_urls
df_news["article_text"] = pd.Series(article_texts, index=df_news.index, dtype="string")
df_news["article_text_length"] = df_news["article_text"].str.len().fillna(0).astype(int)
print(f"Artikel dengan isi berhasil diambil: {(df_news['article_text_length'] >= 100).sum()} dari {len(df_news)}")
print(f"Fetch gagal/kosong: {df_news['article_text'].eq('').sum()}")
df_news = df_news[df_news["article_text_length"] >= 100].reset_index(drop=True)
df_news.to_csv(f"df_news_extended_{mulai}_{mulai + batas}.csv", index=False)

Fetch isi artikel:   0%|          | 0/2473 [00:00<?, ?it/s]

Fetching link: https://news.google.com/rss/articles/CBMi5gFBVV95cUxOUkJyMVhjeUFOSFJjNFJ2Yk5zRUVkcUtzX3ZVQW1rQzRJXzFaNjV4dV9KaUh4a0ZRNFZUVjIyb1JfVm84V0cxR0lFeE5zckpvSmJuOUpOQjNVWVdnNXV5QnFfNlVXQzlBcTBnOHZOWXM1MEhhcHp6YVV5c0Q2cklTSkxfMTBFXzVGaUdDdU4zY01rRUhrVlQ4YU9wZ1hNajZDQnJxN3ZFSGlvNER4dTBzeG5lUF90MmJuS1V3SXRiTFo0MGZIb2QwN1hrUU1wTUJ5U1hyalRReENuaFNRTFA4VXB6YV9wd9IBzAFBVV95cUxPSzBqSHA1RjJ5QzVRcURFNkc3dTU1Tm5pU1hDMlQ0NFN3U0p0SkJCeTV0ak9qdzktRm56Z1BwSElmWEFfdERGcTJUMGFDT0x0M0tGSGFMWnhubmtnVGpLN3d1WmtBazV3Rjl5VHZ4aTBjOGlObUJOVEhEUDJmNWZhZ2pwT0prMjRtTVB3djhBM1JYLUs5Q3JBV1oxYzktQXNrbGFZTGhhVnhzYVVldkRNU1JiaEotb3N5dWN4a29KSHBsMDNYQ21mZDlMaFI?oc=5
panjang artikel: 2162


Fetch isi artikel:  20%|██        | 504/2473 [00:05<00:23, 85.15it/s]

Fetching link: https://news.google.com/rss/articles/CBMizgFBVV95cUxQQXZnbE9WelVKMVNHM3VId3gyM2ZyVDlPV1paeXlHSDhQaTRjZXZmdFU1U0J1cGdOYkQtVV8tdVFUQzByWnNjRkVram1UeEsweFlEMzJQTVhXZGRmSGJtdjJLYlhydWMzZ3RITHlsM2xWTTBJNzlMSjBPdUJyYXF5cDFnX3RJaHo3NXJYcGVTeHdxRW04ZkNGRV9aQUxmZFVjZE1Ec0x2czdnZjZ4VzB6aGNsVnQ4Y0NvVzA5WkVmZXhWRUlBTkRrQ0JaeDljdw?oc=5
panjang artikel: 1523
Fetching link: https://news.google.com/rss/articles/CBMiwAFBVV95cUxOanpQWVVHVGt6ZWdDNFNoWGstN3Q3VmI5WGZwaEl0WFdiem91S1NrTWw0TDl4bElkVGJZeE5makVZTG1QQUFpZ2hIYWdaTm45RVdKd0s2aWUxdnVubVlyWC1JcGNvZXE0Tm9CMGczVS0yb0w0R2FoUHFPZmE2MHVGUzBNM2FlUFFEUGxiMWxseVNzSTQ2Q1hBLVAtcm1jaE1lMVgzcVUyOGNiZ0FBSGRwRmoxS0dqbjlEVHBlSnp4bWY?oc=5
panjang artikel: 2442
Fetching link: https://news.google.com/rss/articles/CBMisAFBVV95cUxOeFVkSVBwU3U1czduLTBONVpjMkxscEN1RjdTcHM1QjRrQk9Vd3l0akV6VzFtN1hWelFldGQ5Nl8tUnFadFJhVldGRk4xQ3hWR20yeHlMOUVkMlRmVHFWRG5ScXNDOTUzb3BpSDMzaFVSdWNCTnBsWllBODZPeGVoLWM0U2RvN3pBNEwyRmlndFlXVHhJVHVoVWQyWnZ6ZGhDOGhack42Yl9jME1aMEkxcdI

Fetch isi artikel:  20%|██        | 504/2473 [00:18<00:23, 85.15it/s]

...

Fetching link: https://news.google.com/rss/articles/CBMisgFBVV95cUxNOTAwOXFfTGpTYlhseC1KZF9RNEpKXy12R29OZzhESlVEWlkzc3VSNTVaTlg2YVRJVjB6S0VBNjVOQ1pPdl94T3BPNm9LbTlCZE9qbmQ1VzExZXItTDJLQzQwSjhJbnlfVWRKdDlkSGhnYVNFT3FUaS04MFp5eVJzbmVBZHp3WnRJYXhaNngtZWlhRDluMEwyZDNRa2NidjR2WVlXYWRWZWNLX3FtQVctckNR?oc=5
panjang artikel: 2020


Fetch isi artikel:  32%|███▏      | 794/2473 [22:02<1:51:10,  3.97s/it]

Fetching link: https://news.google.com/rss/articles/CBMisAFBVV95cUxQa0gxU21jbWVwMHZKNWhKaEtnY2xxckNyeW13ZW9FdW1iazA1WEZmeERBblliLWx2TXlFTDBndnpOa3ZEZVZyOTVvbGZjOVFEMTZOUFhzcjlUWmhfdXVkUGtBeWpPZlVVUlBCLXpDYXY4UHhBZ2tBaHA1NENKV1BuWlI2VkZIT0h4UmwwQjhfM0paVnlsQ3l2SGxORUtKU3BfbmV2VXJ2Sm93bDl4Y1l6WdIBtgFBVV95cUxPYkQ4b3hRWF9xRmE0WnZMRDM3UkhRZkZmZUMzb2xxOWhlVTdLTjFfS2tNY3ZkVjdLMGVoN2g5MDZXSzFDbW1qVEMyQTd3MUVyMnRjYWtTSXNpSHdmTTJxTFowZHFQdW9qbFIyTzBmdlRERktobHpuVk1mYjhBR0lKQ2YwUWNWQkYxTVpIYnNiclZBVFU2Rkg0dllmZFRrb2JxSkpJTXZYTkR2bTNiSEd0MXRPZFc3QQ?oc=5


Fetch isi artikel:  32%|███▏      | 794/2473 [22:08<46:49,  1.67s/it]  


KeyboardInterrupt: 

In [12]:
df_news.to_csv("df_news_extended_503_757.csv", index=False)

In [ ]:
df_news["link"]

0      https://news.google.com/rss/articles/CBMifkFVX...
1      https://news.google.com/rss/articles/CBMitgFBV...
2      https://news.google.com/rss/articles/CBMiwgFBV...
3      https://news.google.com/rss/articles/CBMidEFVX...
4      https://news.google.com/rss/articles/CBMiaEFVX...
                             ...                        
123    https://news.google.com/rss/articles/CBMirwFBV...
124    https://news.google.com/rss/articles/CBMiqgFBV...
125    https://news.google.com/rss/articles/CBMisAFBV...
126    https://news.google.com/rss/articles/CBMiyAFBV...
127    https://news.google.com/rss/articles/CBMizwFBV...
Name: link, Length: 128, dtype: object

In [ ]:
article_texts

['Jumat, 25 September 2026 08:00 UTC Ilustrasi: Pilihan dari Layar. -Dx Gen-AI JATIMNET.COM -\xa0Media sosial mengubah cara memilih tempat makan dalam kehidupan sehari-hari. Restoran yang sebelumnya ditemukan lewat papan nama atau rekomendasi teman kini bisa dikenal melalui video berdurasi beberapa detik. Perubahan tersebut berlangsung di tengah masyarakat Indonesia yang semakin terkoneksi. Data APJII 2025 yang dikutip Kementerian Komunikasi dan Digital mencatat pengguna internet mencapai 229,4 juta orang atau 80,66 persen populasi. Lebih dari 140 juta di antaranya aktif berinteraksi melalui media sosial. Artinya, rekomendasi kuliner tidak lagi bergerak hanya dari mulut ke mulut. Foto, ulasan singkat, video menu, lokasi, hingga komentar pelanggan ikut membentuk keputusan sebelum seseorang berangkat makan. Media Sosial Menjadi Etalase Tempat Makan Dahulu, mencari restoran baru sering dimulai dengan bertanya kepada teman. Cara tersebut masih bertahan, tetapi kini rekomendasinya kerap dil

In [32]:
sample_urls = df_news["link"].head(3)
sample_results = []
for sample_url in sample_urls:
    sample_final_url, sample_html = resolve_and_fetch(sample_url)
    sample_results.append((sample_final_url, extract_article_text(sample_html)))
print("Sampel berhasil diekstrak:", sum(len(text) >= 100 for _, text in sample_results), "dari", len(sample_results))
print("URL sumber pertama:", sample_results[0][0])
print("Panjang teks:", [len(text) for _, text in sample_results])

Sampel berhasil diekstrak: 3 dari 3
URL sumber pertama: https://jatimnet.com/media-sosial-mengubah-cara-kita-memilih-tempat-makan
Panjang teks: [7110, 2855, 6105]


In [33]:
final_urls

['https://jatimnet.com/media-sosial-mengubah-cara-kita-memilih-tempat-makan',
 'https://food.detik.com/rumah-makan/d-8679688/awas-kalap-ini-7-tempat-makan-soto-mie-legendaris-di-jakarta-dan-bogor',
 'https://aktualita.co/ragam/tiktok-food-fest-2026-hari-terakhir-gbk',
 'https://rri.co.id/yogyakarta/kuliner/2752639/rekomendasi-7-tempat-makan-ramah-kantong-di-jogja-dan-tips-menggunakan-grabfood',
 'https://www.nibble.id/tempat-makan-viral-di-bandung-2026/',
 'https://www.tripzilla.id/tempat-makan-di-bekasi-hits-legendaris-dan-ramah-kantong/27718',
 'https://jogja.disway.id/gaya-hidup/read/702022/deretan-warung-ayam-goreng-legendaris-paling-hits-di-purwokerto-dengan-cita-rasa-rempah',
 'https://jatim.idntimes.com/food/dining-guide/10-rekomendasi-kuliner-malam-di-sidoarjo-cocok-bagi-pencinta-kuliner-c1c2-01-njfwj-gnsrbz',
 'https://www.rumah123.com/explore/kota-jakarta-timur/tempat-makan-di-jakarta-timur-yang-murah-dan-enak/',
 'https://erafone.com/artikel/post/wisata-kuliner-malang-04',
 

## 5. Ekstraksi Informasi (Heuristik Regex)

- **Nama tempat**: diambil dari judul berita (bagian sebelum kata penghubung seperti "di", "-", dsb).
- **Lokasi**: dicari pola kata "Kelurahan/Kecamatan/Kabupaten/Kota/Provinsi ..." di teks artikel.
- **Sejak kapan viral**: dicari pola tanggal/bulan di teks, atau memakai tanggal publish berita sebagai perkiraan.
- **Kenapa viral**: kalimat yang memuat frasa "viral karena/usai/setelah/lantaran/gara-gara".

In [ ]:
MONTH_NAMES = (
    "januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember"
)

LOC_PATTERN = re.compile(
    r"(Kelurahan|Kel\.)\s+([A-Z][\w\.\-]*(?:\s+[A-Z][\w\.\-]*){0,3})"
    r"|(Kecamatan|Kec\.)\s+([A-Z][\w\.\-]*(?:\s+[A-Z][\w\.\-]*){0,3})"
    r"|(Kabupaten|Kab\.|Kota)\s+([A-Z][\w\.\-]*(?:\s+[A-Z][\w\.\-]*){0,3})"
    r"|(Provinsi)\s+([A-Z][\w\.\-]*(?:\s+[A-Z][\w\.\-]*){0,3})"
)

VIRAL_SINCE_PATTERN = re.compile(
    rf"(\d{{1,2}}\s+(?:{MONTH_NAMES})\s+\d{{4}}|(?:{MONTH_NAMES})\s+\d{{4}})",
    re.IGNORECASE,
)

VIRAL_REASON_PATTERN = re.compile(
    r"([^.]*\bviral\b[^.]*(?:karena|usai|setelah|lantaran|gara-gara|akibat)[^.]*\.)",
    re.IGNORECASE,
)


def clean_place_candidate(candidate: str) -> str:
    """Ambil nama tempat sebelum deskripsi menu, harga, jam, atau lokasi."""
    candidate = re.sub(r"\s+", " ", candidate).strip(" -:;,\"'")
    candidate = re.split(
        r"\s+(?:yang|dengan|karena|berlokasi|beralamat|terletak|buka|mulai)\s+",
        candidate,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]
    candidate = re.split(r"\s*[.!?]\s*|\s+[—–-]\s+|\s*:\s*", candidate, maxsplit=1)[0]
    candidate = re.split(r",\s*(?:harga|jam|mulai|tersedia|lokasi|alamat)\b", candidate, maxsplit=1, flags=re.IGNORECASE)[0]
    return candidate.strip(" -:;,\"'")


def extract_place_names(text: str) -> list[str]:
    """Ekstrak nama tempat dari marker daftar di awal paragraf atau baris."""
    if not isinstance(text, str) or not text.strip():
        return []

    matches = list(re.finditer(
        r"(?m)^\s*(?:\d{1,2}[.)]|[-•])\s+(.+?)\s*$",
        text,
        flags=re.IGNORECASE,
    ))
    if len(matches) < 2:
        return []

    rejected_terms = (
        "wib", "harga", "rp", "content writer", "copywriter", "baca juga",
        "pertahankan urutan", "daerah istimewa", "dikenal luas", "klik di sini",
        "jam buka", "minggu,", "sabtu,", "senin,", "selasa,", "rabu,",
        "kamis,", "jumat,", "diswayjogja",
    )
    candidates = []
    for match in matches:
        candidate = clean_place_candidate(match.group(1))
        lower_candidate = candidate.casefold()
        if not (3 <= len(candidate) <= 90):
            continue
        if any(term in lower_candidate for term in rejected_terms):
            continue
        if re.search(r"\b\d{1,2}:\d{2}\b|\b\d{1,2}\.\d{2}\b|\b\d{4}\b", candidate):
            continue
        if len(candidate.split()) > 12:
            continue
        candidates.append(candidate)

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        key = candidate.casefold()
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)
    return unique_candidates


def extract_location_text(text: str) -> str:
    """Kumpulkan potongan teks lokasi yang ditemukan dari isi artikel."""
    found = []
    for match in LOC_PATTERN.finditer(text):
        groups = [group for group in match.groups() if group]
        if groups:
            found.append(" ".join(groups))
    seen = set()
    unique = []
    for location in found:
        if location not in seen:
            seen.add(location)
            unique.append(location)
    return "; ".join(unique)


def extract_viral_since(text: str, published) -> str:
    match = VIRAL_SINCE_PATTERN.search(text)
    if match:
        return match.group(1)
    if published is not None:
        return published.strftime("%B %Y") + " (perkiraan dari tanggal berita)"
    return ""


def extract_viral_reason(text: str) -> str:
    match = VIRAL_REASON_PATTERN.search(text)
    return match.group(1).strip() if match else ""


df_news["place_names"] = df_news["article_text"].apply(extract_place_names)
df_news = df_news[df_news["place_names"].map(bool)].drop(columns=["place_name"], errors="ignore").copy()
df_news = df_news.explode("place_names").rename(columns={"place_names": "place_name"}).reset_index(drop=True)
df_news["location_text"] = df_news["article_text"].apply(extract_location_text)
df_news["viral_since"] = df_news.apply(
    lambda row: extract_viral_since(row["article_text"], row["published"]), axis=1
)
df_news["viral_reason"] = df_news["article_text"].apply(extract_viral_reason)

print(f"Total tempat hasil ekstraksi dari isi artikel: {len(df_news)}")
df_news[["place_name", "location_text", "viral_since", "viral_reason"]].head(20)

Total tempat hasil ekstraksi dari isi artikel: 603


,place_name,location_text,viral_since,viral_reason
0,diswayjogja,Kota Purwokerto,September 2026 (perkiraan dari tanggal berita),
1,Dunia kuliner tradisional di Kota Purwokerto t...,Kota Purwokerto,September 2026 (perkiraan dari tanggal berita),
2,diswayjogja,Kota Purwokerto,September 2026 (perkiraan dari tanggal berita),
3,Dunia kuliner tradisional di Kota Purwokerto t...,Kota Purwokerto,September 2026 (perkiraan dari tanggal berita),
4,menghabiskan waktu,,July 2026 (perkiraan dari tanggal berita),
5,menghabiskan waktu,,July 2026 (perkiraan dari tanggal berita),
6,Jam Buka,,22 Mei 2026,
7,"Minggu, 10",,22 Mei 2026,
8,"Minggu, 09",,22 Mei 2026,
9,"Sabtu, 13",,22 Mei 2026,


## 6. Geocoding (Latitude, Longitude, Kelurahan, Kecamatan, Kota/Kab, Provinsi)

Memakai **Nominatim (OpenStreetMap)** dengan `RateLimiter` supaya tidak melanggar batas 1 request/detik dari kebijakan penggunaan Nominatim.

In [ ]:
geolocator = Nominatim(user_agent="riset_tempat_viral_indonesia")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, max_retries=2, error_wait_seconds=2.0)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.1, max_retries=2, error_wait_seconds=2.0)


def geocode_place(place_name: str, location_hint: str):
    """Cari koordinat & detail alamat tempat. Coba dengan nama tempat + hint lokasi, fallback ke hint saja."""
    queries = []
    if location_hint:
        first_hint = location_hint.split(";")[0]
        queries.append(f"{place_name}, {first_hint}, Indonesia")
        queries.append(f"{first_hint}, Indonesia")
    queries.append(f"{place_name}, Indonesia")

    for q in queries:
        try:
            location = geocode(q, addressdetails=True, language="id", country_codes="id")
        except Exception:
            location = None
        if location:
            address = location.raw.get("address", {})
            return {
                "latitude": location.latitude,
                "longitude": location.longitude,
                "kelurahan": address.get("village") or address.get("suburb") or address.get("neighbourhood"),
                "kecamatan": address.get("city_district") or address.get("suburb"),
                "kota_kabupaten": address.get("city") or address.get("county") or address.get("town"),
                "provinsi": address.get("state"),
                "matched_query": q,
            }
    return {
        "latitude": None,
        "longitude": None,
        "kelurahan": None,
        "kecamatan": None,
        "kota_kabupaten": None,
        "provinsi": None,
        "matched_query": None,
    }


geo_results = []
for _, row in tqdm(df_news.iterrows(), total=len(df_news), desc="Geocoding"):
    geo_results.append(geocode_place(row["place_name"], row["location_text"]))

df_geo = pd.DataFrame(geo_results)
df_final = pd.concat([df_news.reset_index(drop=True), df_geo], axis=1)
df_final.head()

## 7. Rapikan & Simpan Hasil Akhir

In [ ]:
output_columns = [
    "place_name",
    "kelurahan",
    "kecamatan",
    "kota_kabupaten",
    "provinsi",
    "latitude",
    "longitude",
    "viral_since",
    "viral_reason",
    "location_text",
    "title",
    "source",
    "published",
    "final_url",
    "keyword",
]

df_result = df_final[output_columns].drop_duplicates(subset=["place_name", "final_url"]).reset_index(drop=True)

df_result.to_csv("tempat_viral_indonesia.csv", index=False, encoding="utf-8-sig")
df_result.to_excel("tempat_viral_indonesia.xlsx", index=False)

print(f"Total data akhir: {len(df_result)}")
df_result

## 8. Langkah Lanjutan (Opsional)

- Tambahkan sumber lain seperti RSS/API dari Detik, Kompas, Tribunnews, atau situs review kuliner (misal: pergikuliner, qraved) untuk memperkaya data.
- Ganti heuristik regex dengan model NER (misal spaCy/IndoBERT) untuk ekstraksi nama tempat & lokasi yang lebih akurat.
- Lakukan verifikasi manual pada kolom `viral_since` dan `viral_reason` karena hasil ekstraksi otomatis bisa kurang tepat pada kalimat yang kompleks.
- Jika butuh volume besar & akurasi geocoding tinggi, pertimbangkan Google Maps Geocoding API (berbayar, ada kuota gratis).